# Tensors: The Engine of Deep Learning — Practical

**ENG. Mohamed Ghannam**

This notebook is the hands-on companion to the lecture *Tensors: The Engine of Deep Learning*.
We move in the same order as the slides:

1. Creating tensors and reading their attributes
2. Data types
3. Reshaping
4. Under the hood: 1D memory and strides
5. NumPy ↔ PyTorch, zero-copy sharing
6. Tensor operations
7. Superpower 1: GPU acceleration (devices & CUDA)
8. Superpower 2: Autograd

Run the cells top to bottom. Each section ends with a short **Observation** that ties the code back to the idea on the slides.

## 0. Setup

PyTorch is imported as `torch`. We fix the random seed so that every run of this notebook produces the same random numbers.

In [167]:
import torch
import numpy as np

print("Using torch", torch.__version__)
torch.manual_seed(42)  # reproducible random numbers

Using torch 2.14.0+cu130


## 1. What is a tensor? Creating tensors

A tensor is an N-dimensional container of numbers:

| Dimensions | Name   | Example                       |
|-----------:|--------|-------------------------------|
| 0D | scalar | a single weight               |
| 1D | vector | a time series                 |
| 2D | matrix | a grayscale image             |
| 3D | cube   | an RGB image (C, H, W)        |
| 4D | batch  | a batch of images (N, C, H, W)|

### 1.1 From existing data with `torch.tensor()`

In [168]:
scalar = torch.tensor(7)                       # 0D
vector = torch.tensor([1, 2, 3])               # 1D
matrix = torch.tensor([[1, 2], [3, 4]])        # 2D
cube   = torch.tensor([[[1, 2], [3, 4]],
                       [[5, 6], [7, 8]]])      # 3D

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix), ("cube", cube)]:
    print(f"{name:7s} ndim = {t.ndim}   shape = {tuple(t.shape)}, ")

scalar  ndim = 0   shape = (), 
vector  ndim = 1   shape = (3,), 
matrix  ndim = 2   shape = (2, 2), 
cube    ndim = 3   shape = (2, 2, 2), 


**Observation:** `ndim` is the number of dimensions and `shape` gives the size of each one. A scalar has `ndim = 0` and an empty shape `()`.

### 1.2 Factory functions

Most of the time we do not have the numbers yet. The `torch` module has a function for every common starting point.

In [169]:
x = torch.empty(3, 4)        # allocates memory but does NOT initialise it (values are garbage)
zeros = torch.zeros(2, 3)    # all zeros
ones = torch.ones(2, 3)      # all ones
rand = torch.rand(2, 3)      # uniform random in [0, 1)
randn = torch.randn(2, 3)    # normal random, mean 0, std 1

print("empty:\n", x)
print("zeros:\n", zeros)
print("ones:\n", ones)
print("rand:\n", rand)
print("randn:\n", randn)

empty:
 tensor([[5.9481e-24, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.2039e-45, 0.0000e+00, 8.4078e-45, 0.0000e+00],
        [1.4013e-44, 0.0000e+00, 4.2039e-44, 0.0000e+00]])
zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
rand:
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])
randn:
 tensor([[ 1.1561,  0.3965, -2.4661],
        [ 0.3623,  0.3765, -0.1808]])


In [170]:
# Ranges
print(torch.arange(5))            # 0 .. 4
print(torch.arange(2, 10, 2))     # start, end (exclusive), step
print(torch.linspace(0, 1, 5))    # 5 evenly spaced numbers from 0 to 1
print(torch.eye(3))               # identity matrix
print(torch.full((2, 2), 9.0))    # filled with one value

tensor([0, 1, 2, 3, 4])
tensor([2, 4, 6, 8])
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor([[9., 9.],
        [9., 9.]])


**Observation:** `torch.empty()` is the fastest because it skips initialisation, but you must overwrite its contents before using them. `torch.rand()` and `torch.randn()` are the usual way to initialise learning weights.

### 1.3 Reproducibility with `torch.manual_seed()`

Random initialisation is everywhere in deep learning, so we need a way to get the *same* random numbers every run.

In [171]:
torch.manual_seed(42)
first = torch.rand(3)

torch.manual_seed(42)
second = torch.rand(3)

print(first)
print(second)
print("identical:", torch.equal(first, second))

tensor([0.8823, 0.9150, 0.3829])
tensor([0.8823, 0.9150, 0.3829])
identical: True


**Observation:** Resetting the seed restarts the random number generator, so the two draws are identical.

### 1.4 Same shape as another tensor: `torch.*_like()`

Operations on two tensors usually need matching shapes. The `*_like` family copies the shape (and dtype, device) of a template tensor.

In [172]:
template = torch.rand(2, 3)

print(template)

print(torch.zeros_like(template))
print(torch.ones_like(template))
print(torch.rand_like(template))
print(torch.empty_like(template).shape)

tensor([[0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411]])
torch.Size([2, 3])


## 2. Tensor attributes

Every tensor carries three pieces of metadata you will read constantly: its **shape**, its **dtype**, and the **device** it lives on.

In [173]:
x = torch.rand(2, 3, 4)

print("shape :", x.shape)          # same as x.size()
print("size  :", x.size())
print("size(0):", x.size(0))       # size of one dimension
print("ndim  :", x.ndim)
print("numel :", x.numel())        # total number of elements = 2 * 3 * 4
print("dtype :", x.dtype)
print("device:", x.device)

shape : torch.Size([2, 3, 4])
size  : torch.Size([2, 3, 4])
size(0): 2
ndim  : 3
numel : 24
dtype : torch.float32
device: cpu


**Observation:** `shape` and `size()` are the same thing. `numel()` is the product of all dimension sizes, which is exactly how many slots the tensor occupies in memory.

## 3. Data types

PyTorch infers the dtype from the data you give it, and lets you set or convert it explicitly.

In [174]:
ints   = torch.tensor([1, 2, 3])
floats = torch.tensor([1.0, 2.0, 3.0])
bools  = torch.tensor([True, False])

print(ints.dtype, floats.dtype, bools.dtype)

torch.int64 torch.float32 torch.bool


In [175]:
# Set the dtype at creation
f32 = torch.tensor([1, 2, 3], dtype=torch.float32)
f64 = torch.tensor([1, 2, 3], dtype=torch.float64)   # "double"
i8  = torch.tensor([1, 2, 3], dtype=torch.int8)

print(f32.dtype, f64.dtype, i8.dtype)

# Convert an existing tensor
print(ints.float().dtype)             # int64  -> float32
print(floats.long().dtype)            # float32 -> int64
print(ints.to(torch.float64).dtype)   # general form

torch.float32 torch.float64 torch.int8
torch.float32
torch.int64
torch.float64


**Observation:** Python integers become `int64` and Python floats become `float32`, PyTorch's default float type. NumPy's default float is `float64`, which is why the shared tensor in the NumPy section below prints `dtype=torch.float64`. Neural network weights are almost always `float32`.

## 4. Reshaping

A tensor with 6 elements can be viewed as `(6,)`, `(2, 3)`, `(3, 2)`, ... as long as the number of elements stays the same.

In [176]:
x = torch.arange(6)
print("x :", x)

v = x.view(2, 3)           # a view: same memory, new shape
r = x.reshape(3, 2)        # reshape: view if possible, otherwise a copy
print("view(2,3):\n", v)
print("reshape(3,2):\n", r)

v[0, 0] = 100              # change the view ...
print("x after editing the view:", x)   # ... and the original changes too

x : tensor([0, 1, 2, 3, 4, 5])
view(2,3):
 tensor([[0, 1, 2],
        [3, 4, 5]])
reshape(3,2):
 tensor([[0, 1],
        [2, 3],
        [4, 5]])
x after editing the view: tensor([100,   1,   2,   3,   4,   5])


In [177]:
x = torch.arange(6)

print(x)
print()
print(x.shape)

print(x.unsqueeze(0))    # add a dimension of size 1 at position 0  -> (1, 6)
print(x.unsqueeze(1).shape)    # add it at position 1                     -> (6, 1)
print(x.unsqueeze(0).squeeze().shape)   # squeeze removes all size-1 dims -> (6,)

tensor([0, 1, 2, 3, 4, 5])

torch.Size([6])
tensor([[0, 1, 2, 3, 4, 5]])
torch.Size([6, 1])
torch.Size([6])


**Observation:** `view()` never copies data. That is only possible because a tensor's shape is *metadata* laid over a flat block of memory, which is exactly the idea of the next section.

## 5. Under the hood: 1D memory and strides

RAM is a single flat line of addresses. A 2D or 3D shape is just a **view** over that line, and the **stride** of each dimension tells PyTorch how many elements to jump to move one step along it.

$$\text{Memory Index} = \text{row}_{idx} \cdot \text{row}_{stride} + \text{col}_{idx} \cdot \text{col}_{stride}$$

In [178]:
x = torch.arange(12).view(3, 4)
print(x)
print("stride:", x.stride())      # (4, 1): one row down = jump 4, one column right = jump 1

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
stride: (4, 1)


In [179]:
# Check the formula from the slides for element (row 2, col 1)
row, col = 2, 1
row_stride, col_stride = x.stride()

memory_index = row * row_stride + col * col_stride
print("memory index:", memory_index)                 # 2*4 + 1*1 = 9
print("x[2, 1]      :", x[row, col].item())
print("flat[9]      :", x.flatten()[memory_index].item())

memory index: 9
x[2, 1]      : 9
flat[9]      : 9


**Observation:** The element at logical position `(2, 1)` is stored at flat address 9, exactly what the formula predicts. PyTorch reports strides in *elements*; NumPy reports them in *bytes*.

### 5.1 Transpose swaps the strides, it does not move data

In [180]:
t = x.T

print("x shape / stride:", tuple(x.shape), x.stride())
print("t shape / stride:", tuple(t.shape), t.stride())     # (1, 4): the strides are swapped

print("same memory block:", x.data_ptr() == t.data_ptr())
print("x contiguous:", x.is_contiguous(), "| t contiguous:", t.is_contiguous())

x shape / stride: (3, 4) (4, 1)
t shape / stride: (4, 3) (1, 4)
same memory block: True
x contiguous: True | t contiguous: False


In [181]:
# A non-contiguous tensor cannot be view()-ed. Uncomment to see the error:
# t.view(12)

# Either make a contiguous copy first ...
t_c = t.contiguous()
print(t_c)
print(t_c.is_contiguous(), t_c.stride())
print(t_c.view(12))

# ... or use reshape(), which copies only when it has to
print(t.reshape(12))

tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
True (3, 1)
tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])
tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])


**Observation:** The transpose is instant because only the stride metadata changed, but the result is *non-contiguous*: walking its memory in order no longer walks the tensor in order. `contiguous()` makes a fresh, properly ordered copy when one is needed.

## 6. NumPy ↔ PyTorch: zero-copy memory sharing

NumPy arrays and CPU tensors use the same flat, C-ordered memory layout, so a tensor can point at an array's memory without copying it.

In [182]:
import numpy as np
import torch

# 1. We create our data in NumPy first
numpy_hallway = np.array([[1.0, 2.0], [3.0, 4.0]])
print("Original NumPy:\n", numpy_hallway)

# ---------------------------------------------------------
# SCENARIO A: Sharing the Memory (Fast, zero-copy)
# ---------------------------------------------------------
# torch.from_numpy() points a PyTorch Tensor at the EXISTING hallway.
shared_tensor = torch.from_numpy(numpy_hallway)

# Let's prove it by changing the first room in the NumPy hallway to 99
numpy_hallway[0, 0] = 99.0

print("\nShared PyTorch Tensor after NumPy update:")
print(shared_tensor) 


# ---------------------------------------------------------
# SCENARIO B: Copying the Memory (Slower, but safe/independent)
# ---------------------------------------------------------
# torch.tensor() builds a BRAND NEW hallway and copies the books over.
independent_tensor = torch.tensor(numpy_hallway)

# Let's change the NumPy hallway again
numpy_hallway[1, 1] = 88.0

print("\nIndependent PyTorch Tensor after NumPy update:")
print(independent_tensor)

Original NumPy:
 [[1. 2.]
 [3. 4.]]

Shared PyTorch Tensor after NumPy update:
tensor([[99.,  2.],
        [ 3.,  4.]], dtype=torch.float64)

Independent PyTorch Tensor after NumPy update:
tensor([[99.,  2.],
        [ 3.,  4.]], dtype=torch.float64)


**Observation:** `torch.from_numpy()` shares memory, so the tensor sees the change to 99 instantly. `torch.tensor()` always copies, so the independent tensor never sees the 88. Sharing is what lets data loaded by OpenCV or Pandas become a tensor with no extra RAM and no copy time.

The link works in the other direction too: `.numpy()` on a CPU tensor also shares memory.

In [183]:
t = torch.ones(3)
n = t.numpy()
print(t)

t.add_(10)          # change the tensor in place ...
print(n)            # ... and the NumPy array shows it: [11. 11. 11.]

tensor([1., 1., 1.])
[11. 11. 11.]


## 7. Tensor operations

PyTorch has over 100 operations. Their behaviour mirrors NumPy, so most of this will feel familiar.

### 7.1 Element-wise arithmetic

In [184]:
x = torch.tensor([[1., 2.],
                  [3., 4.]])

print(x + 2)        # scalar broadcast to every element
print(x * 2)
print(x ** 2)
print(x + x)        # tensor with tensor, same shape
print(x * x)

tensor([[3., 4.],
        [5., 6.]])
tensor([[2., 4.],
        [6., 8.]])
tensor([[ 1.,  4.],
        [ 9., 16.]])
tensor([[2., 4.],
        [6., 8.]])
tensor([[ 1.,  4.],
        [ 9., 16.]])


### 7.2 Shapes must match, unless broadcasting can fix it

Broadcasting compares the shapes from the **last** dimension backwards. Each pair of sizes must be equal, or one of them must be 1, or the dimension must be missing in one tensor.

In [185]:
a = torch.ones(2, 3)
b = torch.tensor([10., 20., 30.]).unsqueeze(0)     # shape (1,3) broadcasts against (2, 3)

print(a)
print()
print(b)
print(b.shape)
print(a + b)

# Shapes (2, 3) and (2,) do NOT line up from the right. Uncomment to see the error:
# a + torch.tensor([1., 2.])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

tensor([[10., 20., 30.]])
torch.Size([1, 3])
tensor([[11., 21., 31.],
        [11., 21., 31.]])


**Observation:** `(2, 3) + (3,)` works because the trailing dimensions match. `(2, 3) + (2,)` fails with *"the size of tensor a must match the size of tensor b at non-singleton dimension 1"*.

### 7.3 Matrix multiplication

In [186]:
A = torch.arange(6.).view(2, 3)
B = torch.arange(6.).view(3, 2)

print(A @ B)                 # (2, 3) @ (3, 2) -> (2, 2)
print(torch.matmul(A, B))    # same thing
print(A.matmul(B))           # same thing
print(A @ A.T)               # transpose to make the shapes compatible

tensor([[10., 13.],
        [28., 40.]])
tensor([[10., 13.],
        [28., 40.]])
tensor([[10., 13.],
        [28., 40.]])
tensor([[ 5., 14.],
        [14., 50.]])


**Observation:** `*` multiplies element by element, `@` does the true matrix product. Mixing the two up is one of the most common beginner bugs.

### 7.4 Indexing and slicing

In [187]:
x = torch.arange(12).view(3, 4)
print(x)

print(x[0])          # first row
print(x[:, 1])       # second column
print(x[:2, -1])     # first two rows, last column
print(x[1:3, :])     # middle two rows
print(x[x > 5])      # boolean mask
print([x > 5])

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([0, 1, 2, 3])
tensor([1, 5, 9])
tensor([3, 7])
tensor([[ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([ 6,  7,  8,  9, 10, 11])
[tensor([[False, False, False, False],
        [False, False,  True,  True],
        [ True,  True,  True,  True]])]


### 7.5 Aggregations

In [188]:
x = torch.arange(12.).view(3, 4)     # float, because mean/std need a floating type

print(x)
print()
print("sum     :", x.sum())
print("mean    :", x.mean())
print("std     :", x.std())
print("mean dim0 (per column):", x.mean(dim=0))
print("mean dim1 (per row)   :", x.mean(dim=1))
print("max / argmax:", x.max(), x.argmax())

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

sum     : tensor(66.)
mean    : tensor(5.5000)
std     : tensor(3.6056)
mean dim0 (per column): tensor([4., 5., 6., 7.])
mean dim1 (per row)   : tensor([1.5000, 5.5000, 9.5000])
max / argmax: tensor(11.) tensor(11)


**Observation:** `dim` names the dimension that is *collapsed*. `mean(dim=0)` averages down the rows and leaves one value per column.

### 7.6 Joining tensors

In [189]:
a = torch.ones(2, 3)
b = torch.zeros(2, 3)

print(a)
print()
print(b)
print()
print(torch.cat([a, b], dim=0))    # stack rows      -> (4, 3)
print(torch.cat([a, b], dim=1).shape)    # stack columns   -> (2, 6)
print()
print(torch.stack([a, b]))         # new leading dim -> (2, 2, 3)

tensor([[1., 1., 1.],
        [1., 1., 1.]])

tensor([[0., 0., 0.],
        [0., 0., 0.]])

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])
torch.Size([2, 6])

tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])


### 7.7 In-place operations

Most operations return a new tensor. The versions ending in an underscore modify the tensor in place.

In [190]:
x = torch.ones(3)

y = x.add(1)      # new tensor, x unchanged
print(x, y)

x.add_(1)         # in place, x itself changes
print(x)

tensor([1., 1., 1.]) tensor([2., 2., 2.])
tensor([2., 2., 2.])


### 7.8 Copying: assignment is not a copy

In [191]:
a = torch.ones(3)
b = a              # b is just another name for the same tensor
c = a.clone()      # c is a real, independent copy

a[0] = 5
print("b:", b)     # sees the change
print("c:", c)     # does not

b: tensor([5., 1., 1.])
c: tensor([1., 1., 1.])


**Observation:** Like Python lists, `b = a` only adds a label. Use `clone()` when you need your own copy of the data.

## 8. Superpower 1: GPU acceleration

Unlike NumPy arrays, tensors can move to a GPU. Moving a tensor physically copies its data over the PCIe bus into the GPU's own memory (VRAM).

In [192]:
import torch

# 1. Ask PyTorch to check the hardware
# "cuda" is the API NVIDIA GPUs use. (Macs use "mps" for Apple Silicon).
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Stadium hired: NVIDIA GPU found!")
else:
    device = torch.device("cpu")
    print("Stuck with the professors: CPU only.")

# 2. Create a tensor on the default CPU memory
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])
print("Currently living on:", cpu_tensor.device)

# 3. Migrate the tensor to the GPU
# This physically copies the data to the GPU's memory chips
gpu_tensor = cpu_tensor.to(device)
print("Now living on:", gpu_tensor.device)

Stadium hired: NVIDIA GPU found!
Currently living on: cpu
Now living on: cuda:0


In [193]:
t = torch.tensor([[1, 2, 3], [6, 10, 30]])
t.device

device(type='cpu')

**Observation:** On a machine without an NVIDIA GPU both tensors stay on `cpu` and the code still runs. That is the point of the `device` variable: write the code once, and it uses whatever hardware is available.

You can also create a tensor directly on the device instead of moving it afterwards.

In [194]:
on_device = torch.ones(3, device=device)
print(on_device.device)

# Rule: tensors in one operation must live on the same device.
# On a CUDA machine this line raises "Expected all tensors to be on the same device":
# cpu_tensor + gpu_tensor

cuda:0


### 8.1 Back to the CPU before NumPy

The copy to the GPU breaks the memory link with NumPy. NumPy only reads system RAM, so a GPU tensor must come back first.

In [195]:
# This will crash! NumPy cannot read VRAM.
# array = gpu_tensor.numpy() 

# The correct way: bring it back to the CPU's memory first, then convert.
cpu_again_tensor = gpu_tensor.cpu()
array = cpu_again_tensor.numpy()

**Observation:** `gpu_tensor.numpy()` fails on a CUDA tensor because there is no CPU memory to share. `.cpu()` copies the data back over the bus, and only then can `.numpy()` work. The one-liner you will see in real code is `tensor.cpu().numpy()`.

## 9. Superpower 2: Autograd

Autograd records every operation on tensors that require gradients, building a directed acyclic graph (DAG) during the forward pass. `backward()` then walks the graph in reverse, applying the chain rule.

For $z = w \cdot x + b$ with $w = 2$, $x = 3$, $b = 1$:

- Forward: $a = w \cdot x = 6$, then $z = a + b = 7$
- Backward: $\dfrac{\partial z}{\partial w} = x = 3$ and $\dfrac{\partial z}{\partial b} = 1$

In [196]:
import torch

# 1. Define the Leaf Nodes (our inputs)
# We set requires_grad=True ONLY for w and b, because in deep learning, 
# we only update the weights and biases, not the input data itself!
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0]) 
b = torch.tensor([1.0], requires_grad=True)

# 2. The Forward Pass (PyTorch builds the DAG dynamically here)
a = w * x       # Creates intermediate tensor 'a' with a MulBackward function
z = a + b       # Creates final tensor 'z' with an AddBackward function

print(f"Forward output z: {z.item()}") 

# 3. The Backward Pass (PyTorch traverses the DAG using the Chain Rule)
z.backward()

# 4. Let's check if PyTorch's math matches our manual calculus!
print(f"Gradient of w (dz/dw): {w.grad.item()}") 
print(f"Gradient of b (dz/db): {b.grad.item()}") 

Forward output z: 7.0
Gradient of w (dz/dw): 3.0
Gradient of b (dz/db): 1.0


**Observation:** The gradients PyTorch stores in `w.grad` and `b.grad` match the hand calculation from the slides. `x` has `requires_grad=False`, so autograd never computes or stores a gradient for it.

We can look at the graph nodes PyTorch attached during the forward pass.

In [197]:
print("a.grad_fn:", a.grad_fn)        # <MulBackward0>  the node that created a
print("z.grad_fn:", z.grad_fn)        # <AddBackward0>  the node that created z
print("w.grad_fn:", w.grad_fn)        # None: w is a leaf, nothing created it

print("leaf tensors:", w.is_leaf, x.is_leaf, b.is_leaf)
print("x.grad:", x.grad)              # None: no gradient requested for x

a.grad_fn: <MulBackward0 object at 0x7fa6bc157f40>
z.grad_fn: <AddBackward0 object at 0x7fa6bc186620>
w.grad_fn: None
leaf tensors: True True True
x.grad: None


**Observation:** Each intermediate tensor remembers the operation that produced it through `grad_fn`. Leaf tensors created by you have no `grad_fn`; they are where the backward pass stops and stores results.

## 10. Summary

- A tensor is an N-dimensional array with a **shape**, a **dtype**, and a **device**.
- Shape is metadata over flat 1D memory. **Strides** map logical positions to memory addresses, which is why `view()` and transpose are free.
- CPU tensors and NumPy arrays can **share memory** (`from_numpy`, `.numpy()`); `torch.tensor()` copies.
- Tensors can move to the **GPU**; that copy breaks the NumPy link, so come back with `.cpu()` first.
- **Autograd** builds a graph in the forward pass and applies the chain rule in `backward()`, storing results in `.grad`.